# Heart Disease Dataset: Import and Preprocessing

This notebook loads `heart.csv` and performs preprocessing in one continuous flow:
- data loading and quick checks
- missing-value and duplicate handling
- feature/target split
- train/test split with stratification
- optional outlier capping and skewness correction
- one-hot encoding for categorical features
- scaling numeric features
- optional feature selection

In [3]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, RobustScaler, StandardScaler

# Load dataset
file_path = "heart.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())

# Basic data quality checks
print("\nMissing values per column:")
print(df.isnull().sum())

duplicate_count = df.duplicated().sum()
print(f"\nDuplicate rows before cleaning: {duplicate_count}")

Dataset shape: (1025, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0



Missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Duplicate rows before cleaning: 723


In [4]:
# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

# Configuration flags for optional techniques
USE_OUTLIER_CAPPING = True
USE_SKEWNESS_CORRECTION = True
USE_ROBUST_SCALER = True
USE_FEATURE_SELECTION = True

# Separate features and target
target_col = "target"
X = df.drop(columns=[target_col])
y = df[target_col]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Define feature groups
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
numerical_features = [col for col in X.columns if col not in categorical_features]

# Work on copies to avoid leakage and keep transformations reproducible
X_train_work = X_train.copy()
X_test_work = X_test.copy()

# Optional 1: IQR-based outlier capping on train, then applied to test
if USE_OUTLIER_CAPPING:
    iqr_bounds = {}
    for col in numerical_features:
        q1 = X_train_work[col].quantile(0.25)
        q3 = X_train_work[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        iqr_bounds[col] = (lower, upper)
        X_train_work[col] = X_train_work[col].clip(lower=lower, upper=upper)
        X_test_work[col] = X_test_work[col].clip(lower=lower, upper=upper)

# Optional 2: skewness-aware power transform on highly skewed numeric columns
skewed_cols = []
if USE_SKEWNESS_CORRECTION:
    skewness = X_train_work[numerical_features].skew().abs()
    skewed_cols = skewness[skewness > 0.75].index.tolist()
    if skewed_cols:
        pt = PowerTransformer(method="yeo-johnson", standardize=False)
        X_train_work.loc[:, skewed_cols] = pt.fit_transform(X_train_work[skewed_cols])
        X_test_work.loc[:, skewed_cols] = pt.transform(X_test_work[skewed_cols])

# Build preprocessing pipelines
scaler = RobustScaler() if USE_ROBUST_SCALER else StandardScaler()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Fit on train and transform train/test
X_train_processed = preprocessor.fit_transform(X_train_work)
X_test_processed = preprocessor.transform(X_test_work)

# Convert to DataFrame for readability
feature_names = preprocessor.get_feature_names_out()
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names)

# Optional 3: feature selection after preprocessing
if USE_FEATURE_SELECTION:
    k_best = min(25, X_train_processed_df.shape[1])
    selector = SelectKBest(score_func=mutual_info_classif, k=k_best)
    X_train_selected = selector.fit_transform(X_train_processed_df, y_train)
    X_test_selected = selector.transform(X_test_processed_df)

    selected_cols = X_train_processed_df.columns[selector.get_support()].tolist()
    X_train_final_df = pd.DataFrame(X_train_selected, columns=selected_cols)
    X_test_final_df = pd.DataFrame(X_test_selected, columns=selected_cols)
else:
    X_train_final_df = X_train_processed_df
    X_test_final_df = X_test_processed_df

# Save outputs
train_output = pd.concat([X_train_final_df.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
test_output = pd.concat([X_test_final_df.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

train_output.to_csv("heart_train_preprocessed.csv", index=False)
test_output.to_csv("heart_test_preprocessed.csv", index=False)

print("\nProcessed train shape:", train_output.shape)
print("Processed test shape:", test_output.shape)
print("Train target distribution:")
print(y_train.value_counts(normalize=True))
print("\nSkewed columns transformed:", skewed_cols if skewed_cols else "None")
print("Saved files:")
print("- heart_train_preprocessed.csv")
print("- heart_test_preprocessed.csv")

display(train_output.head())

Shape after removing duplicates: (302, 14)

Processed train shape: (241, 26)
Processed test shape: (61, 26)
Train target distribution:
target
1    0.543568
0    0.456432
Name: proportion, dtype: float64

Skewed columns transformed: ['oldpeak']
Saved files:
- heart_train_preprocessed.csv
- heart_test_preprocessed.csv


,num__trestbps,num__chol,num__thalach,num__oldpeak,cat__sex_0,cat__sex_1,cat__cp_0,cat__cp_1,cat__cp_2,cat__cp_3,...,cat__ca_0,cat__ca_1,cat__ca_2,cat__ca_3,cat__ca_4,cat__thal_0,cat__thal_1,cat__thal_2,cat__thal_3,target
0,-0.75,2.045455,0.206897,0.441428,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1
1,0.70,-0.621212,-0.965517,0.170157,0.0,1.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
2,-0.10,-0.378788,-0.793103,0.596466,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0
3,0.50,0.196970,-0.275862,0.550197,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0
4,-0.90,-1.227273,-0.551724,-0.558572,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1
